# 1. Load datasets categories

## Sun397

In [ ]:
from datasets import load_dataset
import os

# Load from HuggingFace
data_dir = os.path.join(os.path.dirname(os.getcwd()), "Data", "SUN397")
os.makedirs(data_dir, exist_ok=True)
dataset = load_dataset("tanganke/sun397", cache_dir=data_dir)

# Categories
categories_sun397 = dataset["train"].features["label"].names

print(f"Sun397: {len(categories_sun397)} categories")
for i, cat in enumerate(categories_sun397):
    print(f"{i:3d}  {cat}")

## Places365

In [ ]:
# Open from file
with open("categories_places365.txt", encoding="utf-8") as f:
    lines = f.read().splitlines()

# Clean format
categories_places365 = []
for line in lines:
    line = line.strip()
    if not line:
        continue
    path = line.rsplit(" ", 1)[0]
    name = "/".join(path.split("/")[2:])
    categories_places365.append(name)

print(f"Places365: {len(categories_places365)} categories")
for i, cat in enumerate(categories_places365):
    print(f"{i:3d}  {cat}")

## WIDER

In [ ]:
# Open from file
with open("wider_baseline_cnn_per_class.rst", encoding="utf-8") as f:
    lines = f.read().splitlines()

# Clean format
categories_wider = []
for line in lines:
    line = line.strip()
    if not line:
        continue
    entry = line.split("\t")[0]
    name = entry.split(" ", 1)[1]
    categories_wider.append(name)

print(f"WIDER: {len(categories_wider)} categories")
for i, cat in enumerate(categories_wider):
    print(f"{i:3d}  {cat}")

# 2. Merge categories

In [ ]:
import re
import json

# Normalize
def normalize(name: str) -> str:
    name = name.lower()
    name = name.replace("_", " ").replace("/", " ")
    name = re.sub(r"\s+", " ", name).strip()
    return name

sun_norm = [normalize(c) for c in categories_sun397]
places_norm = [normalize(c) for c in categories_places365]
wider_norm = [normalize(c) for c in categories_wider]

# Merge eliminating duplicates
seen = set()
contexts = []
for cat in sun_norm + places_norm + wider_norm:
    if cat not in seen:
        seen.add(cat)
        contexts.append(cat)

contexts.sort()

n_raw = len(sun_norm) + len(places_norm) + len(wider_norm)
print(f"Raw categories (with duplicates): {n_raw}")
print(f"Unique categories: {len(contexts)}")
print(f"Duplicates deleted: {n_raw - len(contexts)}")

# 3. Save Output

In [ ]:
output = {
    "sources": {
        "sun397": len(sun_norm),
        "places365": len(places_norm),
        "wider": len(wider_norm),
    },
    "total_unique": len(contexts),
    "categories": contexts,
}

with open("contexts.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved in contexts.json")